[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxiRuess/DeepLearning_101/blob/main/notebooks/06_Kernels/03_Matrix_Multiply.ipynb)

# Matrix Multiply — Tiling and Compute-Bound Kernels

This notebook builds on [Vector Add](./01_Vector_Add.ipynb) and [Softmax](./02_Softmax.ipynb) to tackle the **most important kernel in deep learning**: matrix multiplication.

Every `nn.Linear` layer, every attention projection (`Q @ K^T`, `attn @ V`), and every embedding lookup is a matmul. Getting this fast is what makes GPUs useful for deep learning.

| | Softmax (previous) | Matrix Multiply (this notebook) |
|---|---|---|
| Bottleneck | Memory-bound (HBM bandwidth) | **Compute-bound (FLOPs)** |
| Grid | 1D (one program per row) | **2D (one program per output tile)** |
| Key optimization | Fusion (1 pass vs 3) | **Tiling (reuse data in SRAM)** |
| Triton advantage | Eliminates redundant HBM trips | Approaches cuBLAS-level TFLOPS |
| New Triton concepts | `tl.max`, `tl.sum` | **`tl.dot`, 2D grid, K-loop accumulation** |
| Performance metric | Time (microseconds) | **TFLOPS (compute throughput)** |

> Matmul is everywhere in this repo — every `nn.Linear` in the [architecture notebooks](../02_Architectures/) and both `Q @ K^T` and `attn @ V` in the [Flash Attention notebook](../03_Training_Techniques/01_Flash_Attention.ipynb) are matrix multiplications.

## Setup

In [ ]:
import sys, os

# In Colab, clone the repo so local imports (kernels/, src/) work
if "google.colab" in str(get_ipython()):
    if not os.path.exists("/content/DeepLearning_101"):
        !git clone --depth 1 https://github.com/MaxiRuess/DeepLearning_101.git /content/DeepLearning_101
    os.chdir("/content/DeepLearning_101/notebooks/06_Kernels")
    sys.path.insert(0, "/content/DeepLearning_101")
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [ ]:
# Detect runtime environment
import torch

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, '__IPYTHON__') else False
HAS_CUDA = torch.cuda.is_available()

if IN_COLAB:
    %pip install -q triton
    print(f"Running in Colab with GPU: {torch.cuda.get_device_name(0)}")
elif HAS_CUDA:
    print(f"Running locally with GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — will use Modal for remote GPU execution")
    print("Make sure you have Modal configured: pip install modal && modal token set")

## Memory-Bound vs Compute-Bound

Our previous kernels were **memory-bound** — the GPU spent most of its time waiting for data from slow global memory (HBM). Matmul is **compute-bound** — there's so much arithmetic per byte loaded that the bottleneck is the GPU's compute units, not memory bandwidth.

The key metric is **arithmetic intensity**: FLOPs per byte loaded from memory.

| Kernel | FLOPs per element | Bytes loaded per element | Arithmetic intensity |
|---|---|---|---|
| Vector add | 1 | 12 (read 2, write 1) | 0.08 — very low |
| Softmax | ~5 | 8 (read + write) | ~0.6 — low |
| Matmul (naive) | 2K | 8K (load row + col) | ~0.25 — low |
| **Matmul (tiled)** | 2K | 8K / BLOCK | **~BLOCK/4 — high!** |

Tiling is the key: by reusing data already in fast SRAM, we increase arithmetic intensity by a factor of `BLOCK_SIZE`.

```
Naive matmul — one output element at a time:
  For EACH C[i,j]: load row A[i,:] (K floats) + col B[:,j] (K floats)
  Total loads: M * N * 2K = O(MNK) memory ops
  Total compute: M * N * 2K = O(MNK) FLOPs
  → ~1 FLOP per float loaded. GPU compute units sit idle.

Tiled matmul — one output TILE at a time:
  For each BLOCK_M x BLOCK_N tile of C:
    Loop over K in steps of BLOCK_K:
      Load A tile: BLOCK_M * BLOCK_K floats  ┐
      Load B tile: BLOCK_K * BLOCK_N floats   ├ HBM → SRAM (once)
      Compute: BLOCK_M * BLOCK_N * 2*BLOCK_K  │ in SRAM (reused!)
                                               ┘
  Total loads: O(MNK / BLOCK) — reduced by factor of BLOCK!
  Total compute: O(MNK) — same
  → ~BLOCK FLOPs per float loaded. GPU compute units stay busy.
```

## The Tiling Algorithm

We compute `C[M, N] = A[M, K] @ B[K, N]` by dividing C into tiles and accumulating partial results:

```
A [M x K]                B [K x N]               C [M x N]
┌────┬────┬────┬────┐    ┌────┬────┬────┐        ┌────┬────┬────┐
│    │    │    │    │    │    │ B₀ │    │        │    │    │    │
├────┼────┼────┼────┤    ├────┼────┼────┤        ├────┼────┼────┤
│ A₀ │ A₁ │ A₂ │ A₃ │ @ │    │ B₁ │    │   =   │    │ C* │    │
├────┼────┼────┼────┤    ├────┼────┼────┤        ├────┼────┼────┤
│    │    │    │    │    │    │ B₂ │    │        │    │    │    │
└────┴────┴────┴────┘    ├────┼────┼────┤        └────┴────┴────┘
                          │    │ B₃ │    │
                          └────┴────┴────┘

To compute tile C* (one program):
  acc = zeros(BLOCK_M, BLOCK_N)
  acc += A₀ @ B₀     # Load A₀ and B₀ into SRAM, multiply
  acc += A₁ @ B₁     # Load A₁ and B₁ into SRAM, multiply
  acc += A₂ @ B₂     # Load A₂ and B₂ into SRAM, multiply
  acc += A₃ @ B₃     # Load A₃ and B₃ into SRAM, multiply
  C* = acc            # Store final result to HBM
```

| Parameter | Value | Meaning |
|---|---|---|
| `BLOCK_M` | 64 | Rows of C computed per program |
| `BLOCK_N` | 64 | Columns of C computed per program |
| `BLOCK_K` | 32 | Inner dimension tile size |
| Grid | `(ceil(M/64), ceil(N/64))` | 2D grid of programs |
| SRAM per program | ~32 KB | A tile (8KB) + B tile (8KB) + accumulator (16KB) |
| FLOPs per K-step | 262,144 | `64 * 64 * 2 * 32` |

## The Triton Kernel — Explained

The full code lives in `kernels/matmul.py`. Key differences from our previous kernels:

- **2D grid** — `tl.program_id(0)` for tile row, `tl.program_id(1)` for tile column
- **Stride-based 2D addressing** — `offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak` navigates the matrix regardless of memory layout
- **K-loop** — iterates over the inner dimension in tiles, accumulating partial products
- **`tl.dot`** — Triton's tile-level matrix multiply, maps to tensor core instructions on modern GPUs
- **Boundary masking with `0.0`** — out-of-bounds loads return 0, contributing nothing to the dot product (contrast with softmax's `-inf`)

In [ ]:
from kernels.matmul import matmul_kernel, matmul
from pathlib import Path
print(Path("kernels/matmul.py").read_text())

## How the 2D Grid Works

This is our first kernel with a 2D grid:

```
Vector Add: 1D grid over flat array
  Grid = (ceil(N / BLOCK_SIZE),)
  Each program: one chunk of independent elements

Softmax: 1D grid over rows
  Grid = (n_rows,)
  Each program: one full row (reduction)

Matrix Multiply: 2D grid over output tiles
  Grid = (ceil(M / BLOCK_M), ceil(N / BLOCK_N))

  Output C [M x N], divided into tiles:
  ┌─────────┬─────────┬─────────┐
  │ (0, 0)  │ (0, 1)  │ (0, 2)  │
  ├─────────┼─────────┼─────────┤
  │ (1, 0)  │ (1, 1)  │ (1, 2)  │  ← each cell = one program
  ├─────────┼─────────┼─────────┤
  │ (2, 0)  │ (2, 1)  │ (2, 2)  │
  └─────────┴─────────┴─────────┘
  pid_m = tl.program_id(0)  →  tile row
  pid_n = tl.program_id(1)  →  tile column
```

| | Vector Add | Softmax | Matrix Multiply |
|---|---|---|---|
| Grid | 1D | 1D | **2D** |
| Each program | Chunk of elements | One full row | **BLOCK_M x BLOCK_N tile of C** |
| BLOCK_SIZE | Elements per chunk | Must cover row width | **Tile dimensions (M, N, K)** |

## Run on GPU

Triton requires an NVIDIA GPU. This notebook supports two execution modes:
- **Colab / Local CUDA** — runs directly on the available GPU
- **Modal** — runs on a remote T4 GPU (for Mac / no-GPU machines)

In [ ]:
import time

def benchmark_matmul():
    """Run correctness test + benchmark. Works on any CUDA device."""
    import triton
    import triton.language as tl

    @triton.jit
    def _matmul_kernel(
        a_ptr, b_ptr, c_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
        BLOCK_K: tl.constexpr,
    ):
        pid_m = tl.program_id(0)
        pid_n = tl.program_id(1)
        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        a_ptrs = a_ptr + offs_m[:, None] * stride_am + tl.arange(0, BLOCK_K)[None, :] * stride_ak
        b_ptrs = b_ptr + tl.arange(0, BLOCK_K)[:, None] * stride_bk + offs_n[None, :] * stride_bn
        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
        for k in range(0, K, BLOCK_K):
            offs_k = k + tl.arange(0, BLOCK_K)
            a_mask = (offs_m[:, None] < M) & (offs_k[None, :] < K)
            b_mask = (offs_k[:, None] < K) & (offs_n[None, :] < N)
            a_tile = tl.load(a_ptrs, mask=a_mask, other=0.0)
            b_tile = tl.load(b_ptrs, mask=b_mask, other=0.0)
            acc += tl.dot(a_tile, b_tile)
            a_ptrs += BLOCK_K * stride_ak
            b_ptrs += BLOCK_K * stride_bk
        c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
        c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
        tl.store(c_ptrs, acc, mask=c_mask)

    def triton_matmul(a, b):
        M, K = a.shape
        _, N = b.shape
        c = torch.empty((M, N), device=a.device, dtype=a.dtype)
        BLOCK_M, BLOCK_N, BLOCK_K = 64, 64, 32
        grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
        _matmul_kernel[grid](
            a, b, c, M, N, K,
            a.stride(0), a.stride(1),
            b.stride(0), b.stride(1),
            c.stride(0), c.stride(1),
            BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        )
        return c

    # --- Correctness test ---
    torch.manual_seed(0)
    M, N, K = 512, 512, 512
    a = torch.randn(M, K, device="cuda")
    b = torch.randn(K, N, device="cuda")

    output_triton = triton_matmul(a, b)
    output_torch = torch.matmul(a, b)

    max_diff = (output_triton - output_torch).abs().max().item()
    match = torch.allclose(output_triton, output_torch, atol=1e-2)
    print(f"Max difference: {max_diff:.2e}")
    print(f"Results match: {match}")

    # --- Benchmark: vary square matrix size ---
    sizes = [128, 256, 512, 1024, 2048, 4096]
    triton_times = []
    torch_times = []

    for sz in sizes:
        a = torch.randn(sz, sz, device="cuda")
        b = torch.randn(sz, sz, device="cuda")

        for _ in range(10):
            triton_matmul(a, b)
            torch.matmul(a, b)
        torch.cuda.synchronize()

        start = time.perf_counter()
        for _ in range(100):
            triton_matmul(a, b)
        torch.cuda.synchronize()
        triton_times.append((time.perf_counter() - start) / 100)

        start = time.perf_counter()
        for _ in range(100):
            torch.matmul(a, b)
        torch.cuda.synchronize()
        torch_times.append((time.perf_counter() - start) / 100)

    triton_tflops = [2 * sz**3 / (t * 1e12) for sz, t in zip(sizes, triton_times)]
    torch_tflops = [2 * sz**3 / (t * 1e12) for sz, t in zip(sizes, torch_times)]

    return {
        "match": match,
        "max_diff": max_diff,
        "sizes": sizes,
        "triton_us": [t * 1e6 for t in triton_times],
        "torch_us": [t * 1e6 for t in torch_times],
        "triton_tflops": triton_tflops,
        "torch_tflops": torch_tflops,
    }

In [ ]:
if HAS_CUDA:
    # --- Direct GPU execution (Colab or local CUDA) ---
    results = benchmark_matmul()
else:
    # --- Modal remote execution (no local GPU) ---
    import modal

    app = modal.App("triton-matmul")
    image = modal.Image.debian_slim(python_version="3.11").pip_install("torch", "triton")

    @app.function(image=image, gpu="T4")
    def run_remote():
        return benchmark_matmul()

    with app.run():
        results = run_remote.remote()

In [ ]:
if HAS_CUDA:
    # --- Direct GPU execution (Colab or local CUDA) ---
    results = benchmark_matmul()
else:
    # --- Modal remote execution (no local GPU) ---
    # @triton.jit kernels can't be serialized by Modal, so all code is inline.
    import modal

    app = modal.App("triton-matmul")
    image = modal.Image.debian_slim(python_version="3.12").pip_install("torch", "triton")

    @app.function(image=image, gpu="T4")
    def run_remote():
        import torch, triton, triton.language as tl, time

        @triton.jit
        def _matmul_kernel(
            a_ptr, b_ptr, c_ptr, M, N, K,
            stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
            BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
        ):
            pid_m = tl.program_id(0)
            pid_n = tl.program_id(1)
            offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
            offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
            a_ptrs = a_ptr + offs_m[:, None] * stride_am + tl.arange(0, BLOCK_K)[None, :] * stride_ak
            b_ptrs = b_ptr + tl.arange(0, BLOCK_K)[:, None] * stride_bk + offs_n[None, :] * stride_bn
            acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
            for k in range(0, K, BLOCK_K):
                offs_k = k + tl.arange(0, BLOCK_K)
                a_tile = tl.load(a_ptrs, mask=(offs_m[:, None] < M) & (offs_k[None, :] < K), other=0.0)
                b_tile = tl.load(b_ptrs, mask=(offs_k[:, None] < K) & (offs_n[None, :] < N), other=0.0)
                acc += tl.dot(a_tile, b_tile)
                a_ptrs += BLOCK_K * stride_ak
                b_ptrs += BLOCK_K * stride_bk
            c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
            tl.store(c_ptrs, acc, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))

        def triton_matmul(a, b):
            M, K = a.shape
            _, N = b.shape
            c = torch.empty((M, N), device=a.device, dtype=a.dtype)
            BLOCK_M, BLOCK_N, BLOCK_K = 64, 64, 32
            grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
            _matmul_kernel[grid](
                a, b, c, M, N, K,
                a.stride(0), a.stride(1), b.stride(0), b.stride(1),
                c.stride(0), c.stride(1),
                BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
            )
            return c

        torch.manual_seed(0)
        M, N, K = 512, 512, 512
        a = torch.randn(M, K, device="cuda")
        b = torch.randn(K, N, device="cuda")
        out_triton = triton_matmul(a, b)
        out_torch = torch.matmul(a, b)
        max_diff = (out_triton - out_torch).abs().max().item()
        match = torch.allclose(out_triton, out_torch, atol=1e-2)
        print(f"Max difference: {max_diff:.2e}, Results match: {match}")

        sizes = [128, 256, 512, 1024, 2048, 4096]
        triton_times, torch_times = [], []
        for sz in sizes:
            a = torch.randn(sz, sz, device="cuda")
            b = torch.randn(sz, sz, device="cuda")
            for _ in range(10):
                triton_matmul(a, b); torch.matmul(a, b)
            torch.cuda.synchronize()
            start = time.perf_counter()
            for _ in range(100): triton_matmul(a, b)
            torch.cuda.synchronize()
            triton_times.append((time.perf_counter() - start) / 100)
            start = time.perf_counter()
            for _ in range(100): torch.matmul(a, b)
            torch.cuda.synchronize()
            torch_times.append((time.perf_counter() - start) / 100)

        return {
            "match": match, "max_diff": max_diff, "sizes": sizes,
            "triton_us": [t * 1e6 for t in triton_times],
            "torch_us": [t * 1e6 for t in torch_times],
            "triton_tflops": [2*sz**3 / (t*1e12) for sz, t in zip(sizes, triton_times)],
            "torch_tflops": [2*sz**3 / (t*1e12) for sz, t in zip(sizes, torch_times)],
        }

    with app.run():
        results = run_remote.remote()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sizes = results["sizes"]

# --- Plot 1: Execution time ---
ax = axes[0]
ax.plot(sizes, results["torch_us"], "s--", label="torch.matmul (cuBLAS)", linewidth=2, color="#3498db")
ax.plot(sizes, results["triton_us"], "o-", label="Triton", linewidth=2, color="#2ecc71")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Matrix size (N x N)")
ax.set_ylabel("Time (microseconds)")
ax.set_title("Matrix Multiply: Execution Time")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 2: TFLOPS achieved ---
ax = axes[1]
ax.plot(sizes, results["torch_tflops"], "s--", label="torch.matmul (cuBLAS)", linewidth=2, color="#3498db")
ax.plot(sizes, results["triton_tflops"], "o-", label="Triton", linewidth=2, color="#2ecc71")
ax.axhline(y=8.1, color="gray", linestyle=":", alpha=0.5, label="T4 peak (8.1 TFLOPS FP32)")
ax.set_xscale("log", base=2)
ax.set_xlabel("Matrix size (N x N)")
ax.set_ylabel("TFLOPS")
ax.set_title("Matrix Multiply: Compute Throughput")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Why This Kernel Matters for Deep Learning

Matrix multiplication is the computational heart of deep learning:

| Operation | Where it appears | What's happening |
|---|---|---|
| `nn.Linear(in, out)` | Every architecture | `output = input @ weight.T + bias` |
| `Q @ K^T` | Attention | Score matrix computation |
| `attn_weights @ V` | Attention | Weighted value aggregation |
| `embedding @ vocab` | Language models | Token prediction logits |

The **tiling** technique from this kernel is the foundation for [Flash Attention](../03_Training_Techniques/01_Flash_Attention.ipynb), which tiles the entire attention computation (matmul + softmax + matmul) to avoid materializing the $N \times N$ attention matrix in HBM.

## What to Notice

1. **Triton approaches cuBLAS performance** — `torch.matmul` uses NVIDIA's heavily optimized cuBLAS library. Our ~40-line Triton kernel gets surprisingly close, especially at larger sizes where the GPU is fully utilized.

2. **TFLOPS, not just time** — For compute-bound kernels, the right metric is throughput (TFLOPS), not raw time. The T4's theoretical peak is 8.1 TFLOPS for FP32. How close do we get?

3. **Tiling is THE optimization** — Without tiling, every output element loads its own row and column from slow HBM. Tiling reduces HBM accesses from O(MNK) to O(MNK/BLOCK) by reusing data in fast SRAM. This is the same idea behind CPU cache blocking.

4. **First 2D grid** — Vector add and softmax used 1D grids. Matmul uses a 2D grid because the output is a 2D matrix. `tl.program_id(0)` gives the tile row, `tl.program_id(1)` gives the tile column.

5. **`tl.dot` maps to tensor cores** — On NVIDIA GPUs (Volta and later), `tl.dot` compiles to tensor core instructions. The T4 provides 65 TFLOPS for FP16 mixed precision — 8x more than FP32. Production matmul kernels use FP16 inputs with FP32 accumulators to exploit this.

6. **Block sizes matter** — `BLOCK_M=64, BLOCK_N=64, BLOCK_K=32` is tuned for T4 (64KB SRAM). Too small → low arithmetic intensity (too many HBM loads per FLOP). Too large → tiles don't fit in SRAM. Production kernels like cuBLAS auto-tune per GPU.

7. **Looser tolerance than softmax** — We use `atol=1e-2` instead of `1e-6`. Each output element sums K products, and different accumulation orders (Triton tiles vs cuBLAS) cause small differences to accumulate. The relative error is still tiny.

## Resources

- [Triton Matrix Multiplication Tutorial](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html) — Official tutorial with advanced optimizations (L2 cache tiling, split-K)
- [GPU MODE Lectures](https://github.com/gpu-mode/lectures) — Community GPU programming course
- [PMPP Book, Ch. 5](https://www.elsevier.com/books/programming-massively-parallel-processors/hwu/978-0-323-91231-0) — The classic treatment of tiling for matrix multiply
- [How to Optimize a CUDA Matmul Kernel](https://siboehm.com/articles/22/CUDA-MMM) — Step-by-step optimization walkthrough